# Scheduled quantum HUGR generation

Alpha models qubits as first-class linear values over polyhedral domains. This notebook builds a scheduled quantum program, specializes its parameters, and emits a validated HUGR with concrete array boundaries.

In [1]:
import alphalang

QUANTUM_PROGRAM = """affine QuantumPipeline [T,N] -> {:T>0 and N>0}
inputs linear A0, B0 : {[i] : 0 <= i < N} of qubit;
outputs M : {[i] : 0 <= i < N} of bool;
locals linear Q : {[t,i] : 0 <= t < T and 0 <= i < N} of qubit;
       linear A1, B1 : {[i] : 0 <= i < N} of qubit;
let
    over {[t,i] : t=0 and 0<=i<N} with [t,i] : (Q[t,i]) = qalloc();
    over {[t,i] : 0<t<T and 0<=i<N} with [t,i] : (Q[t,i]) = h(Q[t-1,i]);
    with [i] : (M[i]) = measure(Q[T-1,i]);
    with [i] : (A1[i], B1[i]) = cx(A0[i], B0[i]);
    with [i] : () = discard(A1[i]);
    with [i] : () = discard(B1[i]);
.
"""

system = alphalang.parse(QUANTUM_PROGRAM)
normalized = alphalang.normalize(system)
[(variable.name, variable.element_type, variable.multiplicity) for variable in normalized.inputs + normalized.outputs + normalized.locals]

[('A0', ElementType.QUBIT, Multiplicity.LINEAR),
 ('B0', ElementType.QUBIT, Multiplicity.LINEAR),
 ('M', ElementType.BOOL, Multiplicity.UNRESTRICTED),
 ('Q', ElementType.QUBIT, Multiplicity.LINEAR),
 ('A1', ElementType.QUBIT, Multiplicity.LINEAR),
 ('B1', ElementType.QUBIT, Multiplicity.LINEAR)]

## Schedule and specialize

A legal schedule keeps each qubit producer before its consumer. The alternate mapping changes the independent CX chain's placement without changing resource flow.

In [2]:
SCHEDULE = """[T,N] -> {
Q__call0[t,i] -> [t,0,i]; Q__call1[t,i] -> [t,1,i]; M__call0[i] -> [T,2,i];
A1__call0[i] -> [T,3,i]; discard__call0[i] -> [T,4,i]; discard__call1[i] -> [T,5,i]
}"""
ALTERNATE_SCHEDULE = """[T,N] -> {
A1__call0[i] -> [0,0,i]; discard__call0[i] -> [0,1,i]; discard__call1[i] -> [0,2,i];
Q__call0[t,i] -> [t+1,0,i]; Q__call1[t,i] -> [t+1,1,i]; M__call0[i] -> [T+1,2,i]
}"""

scheduled = normalized.schedule(SCHEDULE)
alternate = normalized.schedule(ALTERNATE_SCHEDULE)
print(repr(scheduled))
print("alternate schedule valid:", isinstance(alternate, alphalang.ScheduledSystem))

[T, N] -> { M__call0[i] -> [T, 2, i] : 0 <= i < N; A1__call0[i] -> [T, 3, i] : 0 <= i < N; discard__call0[i] -> [T, 4, i] : 0 <= i < N; discard__call1[i] -> [T, 5, i] : 0 <= i < N; Q__call0[t = 0, i] -> [0, 0, i] : 0 <= i < N; Q__call1[t, i] -> [t, 1, i] : 0 < t < T and 0 <= i < N }
alternate schedule valid: True


In [3]:
bindings = {"T": 3, "N": 4}
envelope = alphalang.generate_hugr(scheduled, bindings)
alternate_envelope = alphalang.generate_hugr(alternate, bindings)
print("prefix:", envelope[:8])
print("characters:", len(envelope))
print("serialized HUGR:", "HUGRiHJ" in envelope)
print("alternate differs:", alternate_envelope != envelope)

prefix: HUGRiHJv
characters: 62487
serialized HUGR: True
alternate differs: True


## Rejected programs

Qubits must be linear, and the compact HUGR realization requires zero-based rectangular resource roots.

In [4]:
NON_LINEAR_QUBIT = """affine Invalid [N] -> {:N>0}
inputs Q : {[i] : 0 <= i < N} of qubit;
outputs M : {[i] : 0 <= i < N} of bool;
let with [i] : (M[i]) = measure(Q[i]);
.
"""

try:
    alphalang.parse(NON_LINEAR_QUBIT)
except ValueError as error:
    print(str(error).splitlines()[-1])

qubit variable 'Q' must be declared linear


In [5]:
TRIANGULAR = """affine Triangular [N] -> {:N>0}
inputs linear Q0 : {[i,j] : 0 <= i < N and 0 <= j <= i} of qubit;
outputs linear Q1 : {[i,j] : 0 <= i < N and 0 <= j <= i} of qubit;
let with [i,j] : (Q1[i,j]) = h(Q0[i,j]);
.
"""

triangular = alphalang.normalize(alphalang.parse(TRIANGULAR))
try:
    alphalang.generate_hugr(triangular, {"N": 3})
except ValueError as error:
    print(error)

realization failed: root domain is not rectangular: { [i, j] : 0 <= i <= 2 and 0 <= j <= i }


## Current boundary

The backend emits concrete HUGRs after all parameters are bound. Measurement-dependent control and parametric HUGR signatures are deferred.